# Turkish ABSA — BERT+ELECTRA Fusion BiLSTM-CRF for Aspect Term Extraction

This notebook contains the training/evaluation code for our proposed aspect term extraction (ATE) architecture: a dual-encoder (BERT + ELECTRA) BiLSTM-CRF model with **convex-combination fusion**, using the **BIOS** tagging scheme and **duplicate** subword label propagation.

This is a focused, single-configuration notebook containing only the final proposed model. The full experimental pipeline used for the paper's ablation studies (tagging-scheme comparison, subword-propagation ablation, fusion-strategy sweep, down-sampling sensitivity, cross-domain evaluation, significance testing) is available in a separate research notebook on request.

**Before running:**
1. Upload the `6k_Turkish_ABSA_Dataset` folder to your Google Drive (see [dataset repository](https://github.com/kevserbusrayildirim/6k_Turkish_ABSA_Dataset)).
2. Adjust `DRIVE_DATA_DIR` and `DRIVE_RESULTS_DIR` below to match your Drive layout.
3. Runtime → Change runtime type → GPU (A100 recommended; any CUDA GPU works with `BATCH_SIZE` adjusted).


In [ ]:
# ============================================================
# 0. COLAB BOOTSTRAP — Drive mount + paket kurulumu
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pytorch-crf seqeval

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU bulunamadi!")
print("CUDA available:", torch.cuda.is_available())


In [ ]:
# ============================================================
# 1. SETUP
# ============================================================
import os, json, random, math, itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, ElectraTokenizerFast, BertModel, ElectraModel
from torchcrf import CRF
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from scipy import stats
from seqeval.metrics import f1_score as seqeval_f1, precision_score as seqeval_precision, recall_score as seqeval_recall, classification_report as seqeval_report

# ---- CONFIG ----
DRIVE_DATA_DIR    = "/content/drive/MyDrive/ABSA/6k_Turkish_ABSA_Dataset-main"   # <-- kendi Drive yolunuza gore duzenleyin
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/ABSA/results"                        # <-- sonuclar Drive'da kalici olarak burada birikir

DATA_DIR = DRIVE_DATA_DIR
TRAIN_CSV = os.path.join(DATA_DIR, "6k_absa_train.csv")
DEV_CSV   = os.path.join(DATA_DIR, "6k_absa_dev.csv")
TEST_CSV  = os.path.join(DATA_DIR, "6k_absa_test.csv")
NEW_SAMPLES_CSV = os.path.join(DATA_DIR, "new_10_samples.csv")   # annotation_helper.py ciktisi; Drive'a yukleyin

BERT_NAME = "dbmdz/bert-base-turkish-cased"
ELECTRA_NAME = "dbmdz/electra-base-turkish-cased-discriminator"

NUM_SEEDS = 5           # A100'de 5 seed makul suredir; sinirli zamanda 3'e dusurun
SEEDS = [13, 42, 2025, 7, 99][:NUM_SEEDS]
EPOCHS = 40
BATCH_SIZE = 32          # A100 icin Kaggle P100'e gore 2x buyutuldu; OOM olursa 16'ya dusurun
NUM_WORKERS = 2          # yuksek RAM'de veri yukleme paralelligi
LR = 2e-5
HIDDEN_DIM = 1024
DROPOUT = 0.4
PATIENCE = 5             # early stopping patience (valid seqeval F1 uzerinden)
IGNORE_INDEX = -100
USE_AMP = True           # A100'de mixed precision ile ~2x hizlanma

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

RESULTS_DIR = DRIVE_RESULTS_DIR
os.makedirs(RESULTS_DIR, exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## 2. Veri Yukleme, Dedup, +10 LLM-Generated Tamamlama ve Leak-Free Re-Split — R1 Comment 6

**Denetim bulgusu (bu revizyon sirasinda tespit edildi)**: orijinal `haziranbirdeneyseti` split'inde (6000 satir) 10 benzersiz metin birden fazla kez geciyordu (20 satir, %0.33): 8'i tam kopya (ayni text+label — muhtemelen split olusturma sirasinda kazara), 2'si ise **ayni ham metnin iki farkli turda farkli annotate edilmesi** (kategori ve/veya span sinirinda uyusmazlik — bu, IAA/annotation-tutarliligi tartismasina somut bir ornek olarak paper'a eklenebilir).

Bu 10 tekrar eden satir cikarilinca 5990 benzersiz satir kaliyor. Dataset'in "6,000 samples" olarak kalmasi icin, az temsil edilen kategorilerden (`education`, `health`, `daily_life`) 10 yeni **LLM-generated** ornek eklendi (`annotation_helper.py` ile uretildi — tokens/labels deterministik/dogrulanmis, sadece text/target/polarity/category LLM kaynakli). Paper'in Conclusion bolumu zaten dataset kaynaklari arasinda "LLM-generated texts"i sayiyordu; Section 3.1 (Collection Process) da bu revizyonda bunu acikca belirtecek sekilde guncellenmeli. Tum satirlara (orijinal + yeni) dogru `source` etiketi (`collected` / `LLM-generated`) atanip birlestiriliyor.

Asagidaki hucre: (1) train+dev+test+yeni-10'u birlestirir, (2) 10 tekrar eden orijinal metnin **tek kopyasini** tutar (5990+10=6000 satir kalir), (3) sabit seed'le (dev=750, test=750, train=kalan 4500) **rastgele** re-split yapar — dedup sonrasi her metin benzersiz oldugu icin split'ler arasinda metin-bazli leakage **yapisal olarak imkansiz** hale gelir. Kategori dagilimi uc split'te de dogal olarak korunuyor (stratify gerekmedi, kontrol amacli yazdiriliyor).

In [ ]:
# ============================================================
# 2. DATA LOADING + DEDUP + 10-SAMPLE TOPUP + LEAK-FREE RE-SPLIT
# ============================================================
_train_raw = pd.read_csv(TRAIN_CSV)
_dev_raw   = pd.read_csv(DEV_CSV)
_test_raw  = pd.read_csv(TEST_CSV)
print(f"Orijinal split -> Train: {len(_train_raw)} | Dev: {len(_dev_raw)} | Test: {len(_test_raw)}")

for _d in (_train_raw, _dev_raw, _test_raw):
    _d["source"] = "collected"

_new_raw = pd.read_csv(NEW_SAMPLES_CSV)  # annotation_helper.py ciktisi, source='LLM-generated' iceriyor
print(f"Yeni (LLM-generated) tamamlama satirlari: {len(_new_raw)}")

full_df = pd.concat([_train_raw, _dev_raw, _test_raw, _new_raw], ignore_index=True)
full_df["_text_norm"] = full_df["text"].astype(str).str.strip().str.lower()
full_df["_labels_str"] = full_df["labels"].astype(str)

text_counts = full_df["_text_norm"].value_counts()
dup_texts = text_counts[text_counts > 1]
exact_dup, inconsistent = 0, 0
for t in dup_texts.index:
    rows = full_df[full_df["_text_norm"] == t]
    if rows["_labels_str"].nunique() == 1:
        exact_dup += 1
    else:
        inconsistent += 1

audit_report = {
    "total_rows_before_dedup": len(full_df),
    "unique_texts": int(full_df["_text_norm"].nunique()),
    "duplicate_texts_found": int(len(dup_texts)),
    "exact_duplicates": exact_dup,
    "annotation_inconsistencies": inconsistent,
}
print("Dedup audit:", audit_report)

# tek kopya birak (dedup) -> 5990 satir
full_df_dedup = full_df.drop_duplicates(subset=["_text_norm"], keep="first").reset_index(drop=True)
print(f"Dedup sonrasi: {len(full_df_dedup)} satir")

# sabit seed'le rastgele re-split (orijinal calismada da stratify kullanilmamisti)
from sklearn.model_selection import train_test_split
DEV_SIZE, TEST_SIZE = 750, 750
train_df, _temp = train_test_split(full_df_dedup, test_size=(DEV_SIZE + TEST_SIZE), random_state=42)
dev_df, test_df  = train_test_split(_temp, test_size=TEST_SIZE, random_state=42)
train_df, dev_df, test_df = (d.reset_index(drop=True) for d in (train_df, dev_df, test_df))
print(f"Yeni split -> Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")

for name, d in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    print(f"{name} category dagilimi (top 5):")
    print(d["category"].value_counts(normalize=True).head().to_string())

def _norm_text_set(df):
    return set(df["_text_norm"])

leakage_report = {
    "train_dev_overlap": len(_norm_text_set(train_df) & _norm_text_set(dev_df)),
    "train_test_overlap": len(_norm_text_set(train_df) & _norm_text_set(test_df)),
    "dev_test_overlap": len(_norm_text_set(dev_df) & _norm_text_set(test_df)),
}
print("Leakage check (re-split sonrasi):", leakage_report)
assert sum(leakage_report.values()) == 0, "UYARI: dedup sonrasi hala overlap var, re-split mantigini kontrol edin!"

with open(os.path.join(RESULTS_DIR, "leakage_audit.json"), "w") as f:
    json.dump({**audit_report, **leakage_report}, f, indent=2)

for d in (train_df, dev_df, test_df):
    d.drop(columns=["_text_norm", "_labels_str"], inplace=True, errors="ignore")

train_df.to_csv(os.path.join(RESULTS_DIR, "6k_absa_train_v2.csv"), index=False)
dev_df.to_csv(os.path.join(RESULTS_DIR, "6k_absa_dev_v2.csv"), index=False)
test_df.to_csv(os.path.join(RESULTS_DIR, "6k_absa_test_v2.csv"), index=False)

# tokens / labels kolonlarini list'e cevir
for df in (train_df, dev_df, test_df):
    for col in ("tokens", "labels"):
        if df[col].dtype == object:
            df[col] = df[col].apply(lambda x: eval(x) if isinstance(x, str) else x)


## 3. Etiket Semasi Donusumu + Subword Label Propagation — R1 Comments 5, 10

**Onemli duzeltme**: veri kumesindeki `labels` sutunu HER ZAMAN BIOS formatinda saklaniyor (S etiketiyle). BIO/BIOES semalarini denemek icin once bu ham BIOS etiketlerini hedef semaya donusturen bir adim gerekiyor (`convert_bios_to_scheme`) — bu adim eksik oldugunda BIO semasi denerken tag2idx'te olmayan 'S' anahtarina rastlanip `KeyError` aliniyordu. Donusum kurallari:
- **BIO**: S -> B (tekil aspect de B ile temsil edilir, ardindan I gelmez)
- **BIOS**: degisiklik yok (zaten native format)
- **BIOES**: coklu-kelimeli span'in SON 'I'si -> E (span'in nerede bittigini isaretler); B ve S oldugu gibi kalir

Ayrica subword propagation (`align_labels`) da E etiketini dogru ele alacak sekilde guncellendi: bir kelime E etiketliyken subword'lere bolunurse, SON subword E'yi tasir, ONCEKI subword'ler I alir (B/S icin ilk subword'un ozel isaret tasimasinin simetrigi). 3 propagation stratejisi (hepsi ablation icin kullanilabilir):

- `duplicate`: eski davranış (kontrol/baseline olarak tutuluyor)
- `first_marks_rest_i`: B/S ilk subword'de tutulur kalanlar I; E son subword'de tutulur oncekiler I; I/O tum subword'lerde ayni kalir
- `first_only_mask_rest`: standart NER pratiği — sadece ilk subword gerçek etiketi alır, kalan subword'ler loss'tan `IGNORE_INDEX=-100` ile hariç tutulur (CRF bu durumda mask ile kullanılır)

In [ ]:
# ============================================================
# 3. LABEL SCHEME CONVERSION + SUBWORD LABEL ALIGNMENT
# ============================================================
def convert_bios_to_scheme(labels, scheme):
    """Veri kumesindeki HAM BIOS etiketlerini (O,B,I,S) hedef semaya cevirir.
    labels: kelime-seviyesi BIOS etiket listesi (subword bolme ONCESI)."""
    if scheme == "BIOS":
        return list(labels)
    if scheme == "BIO":
        return ["B" if l == "S" else l for l in labels]
    if scheme == "BIOES":
        n = len(labels)
        out = []
        for i, l in enumerate(labels):
            if l == "I":
                is_last_of_span = (i == n - 1) or (labels[i + 1] != "I")
                out.append("E" if is_last_of_span else "I")
            else:
                out.append(l)
        return out
    raise ValueError(f"Bilinmeyen scheme: {scheme}")


def align_labels(labels, word_ids, tag2idx, strategy="first_marks_rest_i"):
    """labels: convert_bios_to_scheme'den GECMIS, hedef semaya ait kelime-seviyesi
    etiketler (BIO/BIOS/BIOES). word_ids: tokenizer.word_ids() cikisi (None =
    special token, CLS/SEP).

    Onemli: CLS/SEP her zaman tag2idx['O'] alir (strateji ne olursa olsun) ve
    HICBIR ZAMAN IGNORE_INDEX olmaz. Nedeni: torchcrf.CRF, mask[:,0]'in (dizinin
    ilk pozisyonu = CLS) mutlaka True olmasini zorunlu kilar; CLS'i IGNORE_INDEX
    yapip crf_mask'te False'a dusurmek runtime'da assertion hatasi verir.
    first_only_mask_rest stratejisindeki 'mask' sadece bir KELIMENIN ilk
    subword'unden SONRAKI subword'lerine uygulanir, CLS/SEP'e degil."""
    new_labels = []
    n = len(word_ids)
    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            new_labels.append(tag2idx["O"])
            continue

        orig_label = labels[word_idx]
        is_first_subword = (idx == 0) or (word_ids[idx - 1] != word_idx)
        is_last_subword = (idx == n - 1) or (word_ids[idx + 1] != word_idx)

        if strategy == "duplicate":
            new_labels.append(tag2idx[orig_label])

        elif strategy == "first_marks_rest_i":
            if orig_label in ("B", "S"):
                new_labels.append(tag2idx[orig_label] if is_first_subword else tag2idx["I"])
            elif orig_label == "E":
                new_labels.append(tag2idx[orig_label] if is_last_subword else tag2idx["I"])
            else:  # 'I' veya 'O'
                new_labels.append(tag2idx[orig_label])

        elif strategy == "first_only_mask_rest":
            if is_first_subword:
                new_labels.append(tag2idx[orig_label])
            else:
                new_labels.append(IGNORE_INDEX)
        else:
            raise ValueError(f"Bilinmeyen strateji: {strategy}")

    return new_labels


In [ ]:
# ============================================================
# 4. DATASET / COLLATE  (DUZELTILDI: first_subword_mask / word_mask eklendi)
# ============================================================
class NerDataset(Dataset):
    """encoders: {'bert': (tokenizer, name)} veya {'bert':..., 'electra':...} - tek/cift encoder destekler.
    scheme: 'BIO'|'BIOS'|'BIOES' - df'teki ham BIOS etiketleri bu hedefe cevrilir."""
    def __init__(self, df, encoders, tag2idx, strategy, scheme="BIOS"):
        self.df = df.reset_index(drop=True)
        self.encoders = encoders
        self.tag2idx = tag2idx
        self.strategy = strategy
        self.scheme = scheme

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        words, raw_labels = row["tokens"], row["labels"]
        scheme_labels = convert_bios_to_scheme(raw_labels, self.scheme)
        item = {"words": words}
        ref_word_ids = None
        for key, tok in self.encoders.items():
            enc = tok(words, is_split_into_words=True, return_attention_mask=False, truncation=True)
            item[f"input_ids_{key}"] = enc["input_ids"]
            if ref_word_ids is None:
                ref_word_ids = enc.word_ids()
        item["labels"] = align_labels(scheme_labels, ref_word_ids, self.tag2idx, self.strategy)

        # KRITIK DUZELTME: strateji ne olursa olsun (duplicate/first_marks_rest_i/
        # first_only_mask_rest), HANGI pozisyonlarin bir kelimenin ILK subword'u
        # oldugunu ayri bir maskede tutuyoruz. Nedeni: crf_mask sadece
        # first_only_mask_rest'te kelime-seviyesine indirgeme yapiyordu; diger
        # iki stratejide TUM subword'ler skorlaniyordu, bu da duplicate/
        # first_marks_rest_i icin span-F1'i tutarsiz sekilde etkiliyordu
        # (bkz. sohbet gecmisi). Artik evaluate() TUM stratejilerde bu maskeyi
        # kullanip sadece ilk subword'leri skorlayacak.
        first_subword_mask = []
        prev = None
        for wid in ref_word_ids:
            first_subword_mask.append(wid is not None and wid != prev)
            prev = wid
        item["first_subword_mask"] = first_subword_mask

        # KRITIK: BERT ve ELECTRA tokenizer'lari ayni cumle icin farkli sayida
        # subword uretebilir (farkli vocab/segmentation). Fusion modelde iki
        # encoder'in ciktisi pozisyon-bazli birlestirildigi (concat/average/gated/
        # learned/convex) icin her iki input_ids dizisinin de labels ile AYNI
        # uzunlukta olmasi ZORUNLU. Burada labels uzunluguna gore per-sample
        # kirp/pad yapiliyor (kirpma cok nadir olmali; olursa saydigimiz sayaci
        # asagida logluyoruz).
        target_len = len(item["labels"])
        for key, tok in self.encoders.items():
            seq = item[f"input_ids_{key}"]
            if len(seq) > target_len:
                seq = seq[:target_len]
            elif len(seq) < target_len:
                pad_id = tok.pad_token_id or 0
                seq = seq + [pad_id] * (target_len - len(seq))
            item[f"input_ids_{key}"] = seq
        return item

def make_collate_fn(encoders, tag2idx):
    def collate_fn(batch):
        batch = [b for b in batch if b is not None]
        words = [b["words"] for b in batch]
        max_len = max(len(b["labels"]) for b in batch)
        out = {"words": words}
        for key, tok in encoders.items():
            pad_id = tok.pad_token_id or 0
            ids = [b[f"input_ids_{key}"] for b in batch]
            ids_padded = [seq + [pad_id] * (max_len - len(seq)) for seq in ids]
            out[f"input_ids_{key}"] = torch.tensor(ids_padded, dtype=torch.long)
        labels = [b["labels"] for b in batch]
        labels_padded = [seq + [IGNORE_INDEX] * (max_len - len(seq)) for seq in labels]
        out["labels"] = torch.tensor(labels_padded, dtype=torch.long)
        first_key = next(iter(encoders))
        attn = [[1]*len(b[f"input_ids_{first_key}"]) + [0]*(max_len - len(b[f"input_ids_{first_key}"])) for b in batch]
        out["attention_mask"] = torch.tensor(attn, dtype=torch.bool)
        # CRF mask: IGNORE_INDEX konumlarinda da CRF'e gormezden gelinmesi icin ayri bir mask
        crf_mask = (out["labels"] != IGNORE_INDEX) & out["attention_mask"]
        out["crf_mask"] = crf_mask
        # YENI: kelime-seviyesi skorlama icin first-subword maskesi (tum stratejilerde tutarli)
        wm = [b["first_subword_mask"] + [False] * (max_len - len(b["first_subword_mask"])) for b in batch]
        out["word_mask"] = torch.tensor(wm, dtype=torch.bool)
        return out
    return collate_fn


## 5. Model Mimarileri

`learned` fusion, paper'daki Eşitlik (7)-(9)'da tarif edildiği gibi **ayrı** `W_bert`, `W_electra` matrisleriyle implement edildi (eski koddaki tek `Linear(2d, d)` katmanı matematiksel olarak eşdeğer ama ayrı ağırlık normlarını loglamak imkansızdı — Reviewer 1 Comment 4 tam bunu istiyor). Ayrıca **`convex`** adında yeni bir fusion stratejisi eklendi: `alpha = sigmoid(scalar)` ile `alpha*h_bert + (1-alpha)*h_electra` — reviewer'ın önerdiği "daha ilkeli convex combination" alternatifi.

**CRF + ic-bosluklu maske duzeltmesi (kritik)**: `first_only_mask_rest` stratejisinde `crf_mask` dizinin ORTASINDA bosluklar icerir (maskelenmis ara subword'ler). `pytorch-crf`'in egitim kaybi (`_compute_score`) bu bosluklari YANLIS ele aliyor — bir sonraki gecerli pozisyona gecis skorunu hesaplarken, bosluktaki DUMMY etiketi ('O') gercek onceki etiketmis gibi kullaniyor, bu da sahte gecis skorlari uretip egitimi ciddi sekilde bozuyor (`decode()` bunu dogru marjinalize ediyor ama egitim kaybi yapmiyor - asimetrik bug). Cozum: CRF'e vermeden ONCE, `crf_mask`'teki True pozisyonlari sola yaslayip (compact) tamamen surekli/boslugsuz bir diziye indirgemek. Diger 2 stratejide (`duplicate`, `first_marks_rest_i`) maskede zaten ic bosluk olmadigi icin bu islem NO-OP'tur (sonuclari degistirmez), sadece `first_only_mask_rest`'i duzeltir.

In [ ]:
# ============================================================
# 5. MODELS
# ============================================================
def _compact_for_crf(tensor_2d_or_3d, mask):
    """tensor: (B,T,H) veya (B,T); mask: (B,T) bool. Her ornek icin True
    pozisyonlari sola yaslar (compact), yeni surekli (gap'siz) mask doner.
    CRF'e HER ZAMAN bu compact hallerle girilir -- boylece pytorch-crf hicbir
    zaman ic-bosluklu bir maskeyle karsilasmaz (bkz. yukaridaki not)."""
    B, T = mask.shape
    lengths = mask.sum(dim=1)
    max_len = max(1, int(lengths.max().item()))
    is_3d = tensor_2d_or_3d.dim() == 3
    if is_3d:
        H = tensor_2d_or_3d.size(-1)
        out = tensor_2d_or_3d.new_zeros(B, max_len, H)
    else:
        out = tensor_2d_or_3d.new_zeros(B, max_len)
    new_mask = torch.zeros(B, max_len, dtype=torch.bool, device=mask.device)
    for b in range(B):
        idx = mask[b].nonzero(as_tuple=True)[0]
        L = idx.numel()
        if L == 0:
            continue
        out[b, :L] = tensor_2d_or_3d[b, idx]
        new_mask[b, :L] = True
    return out, new_mask


class SingleEncoderBiLSTMCRF(nn.Module):
    """BERT-only veya ELECTRA-only baseline."""
    def __init__(self, encoder_name, num_tags, hidden_dim=HIDDEN_DIM, dropout=DROPOUT, encoder_type="bert"):
        super().__init__()
        self.encoder = BertModel.from_pretrained(encoder_name) if encoder_type == "bert" else ElectraModel.from_pretrained(encoder_name)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(768, hidden_dim // 2, num_layers=2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags=num_tags, batch_first=True)

    def _emissions(self, input_ids, attention_mask):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        lstm_out, _ = self.lstm(self.dropout(h))
        return self.fc(self.dropout(lstm_out))

    def forward(self, input_ids, attention_mask, crf_mask):
        emissions = self._emissions(input_ids, attention_mask)
        compact_emissions, compact_mask = _compact_for_crf(emissions, crf_mask)
        return self.crf.decode(compact_emissions, mask=compact_mask)

    def neg_log_likelihood(self, input_ids, tags, attention_mask, crf_mask):
        emissions = self._emissions(input_ids, attention_mask)
        safe_tags = tags.clone(); safe_tags[safe_tags == IGNORE_INDEX] = 0
        compact_emissions, compact_mask = _compact_for_crf(emissions, crf_mask)
        compact_tags, _ = _compact_for_crf(safe_tags, crf_mask)
        return -self.crf(compact_emissions, compact_tags, mask=compact_mask, reduction='mean')


class FusionBiLSTMCRF(nn.Module):
    """fusion in {'concat','average','gated','learned','convex'}"""
    def __init__(self, bert_name, electra_name, num_tags, fusion="learned",
                 hidden_dim=HIDDEN_DIM, dropout=DROPOUT, enc_dim=768):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_name)
        self.electra = ElectraModel.from_pretrained(electra_name)
        self.dropout = nn.Dropout(dropout)
        self.fusion = fusion
        self.enc_dim = enc_dim
        self.last_weight_norms = None  # interpretability logging icin

        if fusion == "concat":
            fusion_dim = enc_dim * 2
        elif fusion == "average":
            fusion_dim = enc_dim
        elif fusion == "gated":
            fusion_dim = enc_dim
            self.gating_layer = nn.Linear(enc_dim * 2, enc_dim)
        elif fusion == "learned":
            fusion_dim = enc_dim
            self.w_bert = nn.Linear(enc_dim, enc_dim)
            self.w_electra = nn.Linear(enc_dim, enc_dim)
        elif fusion == "convex":
            fusion_dim = enc_dim
            self.alpha_raw = nn.Parameter(torch.tensor(0.0))  # sigmoid(0)=0.5 ile baslar
        else:
            raise ValueError(f"Bilinmeyen fusion: {fusion}")

        self.lstm = nn.LSTM(fusion_dim, hidden_dim // 2, num_layers=2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags=num_tags, batch_first=True)

    def _fuse(self, h_bert, h_electra):
        if self.fusion == "concat":
            return torch.cat((h_bert, h_electra), dim=-1)
        if self.fusion == "average":
            return (h_bert + h_electra) / 2
        if self.fusion == "gated":
            gates = torch.sigmoid(self.gating_layer(torch.cat((h_bert, h_electra), dim=-1)))
            return gates * h_bert + (1 - gates) * h_electra
        if self.fusion == "learned":
            t_bert = self.w_bert(h_bert)
            t_electra = self.w_electra(h_electra)
            # interpretability: her forward'da encoder katkisinin L2 normunu logla
            self.last_weight_norms = {
                "w_bert_norm": self.w_bert.weight.norm().item(),
                "w_electra_norm": self.w_electra.weight.norm().item(),
                "bert_activation_norm": t_bert.detach().norm(dim=-1).mean().item(),
                "electra_activation_norm": t_electra.detach().norm(dim=-1).mean().item(),
            }
            return t_bert + t_electra
        if self.fusion == "convex":
            alpha = torch.sigmoid(self.alpha_raw)
            self.last_weight_norms = {"alpha_bert": alpha.item(), "alpha_electra": (1 - alpha).item()}
            return alpha * h_bert + (1 - alpha) * h_electra
        raise RuntimeError

    def _emissions(self, input_ids_bert, input_ids_electra, attention_mask):
        h_bert = self.bert(input_ids=input_ids_bert, attention_mask=attention_mask).last_hidden_state
        h_electra = self.electra(input_ids=input_ids_electra, attention_mask=attention_mask).last_hidden_state
        fused = self._fuse(h_bert, h_electra)
        lstm_out, _ = self.lstm(self.dropout(fused))
        return self.fc(self.dropout(lstm_out))

    def forward(self, input_ids_bert, input_ids_electra, attention_mask, crf_mask):
        emissions = self._emissions(input_ids_bert, input_ids_electra, attention_mask)
        compact_emissions, compact_mask = _compact_for_crf(emissions, crf_mask)
        return self.crf.decode(compact_emissions, mask=compact_mask)

    def neg_log_likelihood(self, input_ids_bert, input_ids_electra, tags, attention_mask, crf_mask):
        emissions = self._emissions(input_ids_bert, input_ids_electra, attention_mask)
        safe_tags = tags.clone(); safe_tags[safe_tags == IGNORE_INDEX] = 0
        compact_emissions, compact_mask = _compact_for_crf(emissions, crf_mask)
        compact_tags, _ = _compact_for_crf(safe_tags, crf_mask)
        return -self.crf(compact_emissions, compact_tags, mask=compact_mask, reduction='mean')


## 6. Evaluation — Word-level + seqeval Span-level (R1 Comment 9)

Eski `evaluate_model` fonksiyonu zaten subword'leri kelime bazında tekilleştiriyordu (yani raporlanan "token accuracy" aslında **word-level** idi, tam olarak reviewer'ın varsaydığı gibi ham subword-token accuracy değildi) — ama yine de bu word-level metrik, dominant O etiketinden (%74) şişebiliyor. Bu yüzden **seqeval span-level exact-match F1** ekleniyor; bu, NER/ATE literatüründeki kabul görmüş standart ve prior work ile karşılaştırılabilir olacak.

In [ ]:
# ============================================================
# 6. EVALUATION  (DUZELTILDI: sadece first-subword pozisyonlari skorlanir)
# ============================================================
def _bio_normalize(tags):
    """CRF/propagation ciktisi BIOS/BIOES olabilir; seqeval BIO bekler.
    S -> B (tekil span), E -> I (span sonu, BIO'da ayrim yok)."""
    mapping = {"S": "B", "E": "I"}
    return [mapping.get(t, t) for t in tags]

def evaluate(model, data_loader, idx2tag, device, dual_encoder=True):
    model.eval()
    y_true, y_pred = [], []
    weight_logs = []
    with torch.no_grad():
        for batch in data_loader:
            attention_mask = batch["attention_mask"].to(device)
            crf_mask = batch["crf_mask"].to(device)
            word_mask = batch["word_mask"].to(device)
            labels = batch["labels"]

            if dual_encoder:
                ids_bert = batch["input_ids_bert"].to(device)
                ids_electra = batch["input_ids_electra"].to(device)
                preds = model(ids_bert, ids_electra, attention_mask, crf_mask)
            else:
                ids = batch["input_ids_bert"].to(device)
                preds = model(ids, attention_mask, crf_mask)

            if getattr(model, "last_weight_norms", None):
                weight_logs.append(model.last_weight_norms)

            for b in range(labels.size(0)):
                mask_row = crf_mask[b].cpu().numpy()
                word_row = word_mask[b].cpu().numpy()
                true_row = labels[b].numpy()
                pred_row = preds[b]
                true_tags, pred_tags = [], []
                pi = 0
                for pos, keep in enumerate(mask_row):
                    if not keep:
                        continue
                    if word_row[pos]:
                        true_tags.append(idx2tag[true_row[pos]])
                        pred_tags.append(idx2tag[pred_row[pi]])
                    pi += 1
                y_true.append(true_tags)
                y_pred.append(pred_tags)

    flat_true = [t for seq in y_true for t in seq]
    flat_pred = [t for seq in y_pred for t in seq]

    word_acc = accuracy_score(flat_true, flat_pred)
    word_f1_macro = f1_score(flat_true, flat_pred, average="macro", zero_division=0)

    y_true_bio = [_bio_normalize(seq) for seq in y_true]
    y_pred_bio = [_bio_normalize(seq) for seq in y_pred]
    span_f1 = seqeval_f1(y_true_bio, y_pred_bio)
    span_precision = seqeval_precision(y_true_bio, y_pred_bio)
    span_recall = seqeval_recall(y_true_bio, y_pred_bio)

    result = {
        "word_accuracy": word_acc,
        "word_f1_macro": word_f1_macro,
        "span_f1": span_f1,
        "span_precision": span_precision,
        "span_recall": span_recall,
    }
    if weight_logs:
        keys = weight_logs[0].keys()
        for k in keys:
            result[f"mean_{k}"] = float(np.mean([w[k] for w in weight_logs]))
    return result, (y_true, y_pred)


## 7. Eğitim Döngüsü + Multi-Seed Runner (R1 Comment 8)

Her konfigürasyon `NUM_SEEDS` farklı seed ile koşulur; ortalama ± std raporlanır. İki konfigürasyon arasında **paired t-test** (`compare_configs`) ile anlamlılık test edilir — paper'da "learnable fusion consistently best" iddiası bu testle desteklenmeli ya da yumuşatılmalı.

In [ ]:
# ============================================================
# 7. TRAINING LOOP + MULTI-SEED RUNNER
# ============================================================
def train_one_run(model, train_loader, valid_loader, idx2tag, device, dual_encoder=True,
                   epochs=EPOCHS, lr=LR, patience=PATIENCE, verbose=False, use_amp=USE_AMP):
    # A100: bf16 autocast kullaniyoruz (fp16'nin aksine GradScaler gerektirmez,
    # CRF'in logsumexp gibi islemlerinde underflow riski dusuk kalir).
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_f1, best_state, no_improve = -1, None, 0
    amp_enabled = use_amp and torch.cuda.is_available()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            attention_mask = batch["attention_mask"].to(device)
            crf_mask = batch["crf_mask"].to(device)
            labels = batch["labels"].to(device)
            optimizer.zero_grad()
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=amp_enabled):
                if dual_encoder:
                    ids_bert = batch["input_ids_bert"].to(device)
                    ids_electra = batch["input_ids_electra"].to(device)
                    loss = model.neg_log_likelihood(ids_bert, ids_electra, labels, attention_mask, crf_mask)
                else:
                    ids = batch["input_ids_bert"].to(device)
                    loss = model.neg_log_likelihood(ids, labels, attention_mask, crf_mask)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        valid_metrics, _ = evaluate(model, valid_loader, idx2tag, device, dual_encoder)
        if verbose:
            print(f"  epoch {epoch+1}: loss={total_loss/len(train_loader):.4f} valid_span_f1={valid_metrics['span_f1']:.4f}")

        if valid_metrics["span_f1"] > best_f1:
            best_f1 = valid_metrics["span_f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose:
                    print(f"  early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    return model, best_f1


def run_multi_seed(model_fn, train_loader, valid_loader, test_loader, idx2tag, device,
                    dual_encoder=True, seeds=SEEDS, epochs=EPOCHS, verbose=False):
    """model_fn: seed -> yeni model instance (her seed icin sifirdan init)."""
    per_seed_results = []
    for seed in seeds:
        set_seed(seed)
        model = model_fn()
        model, _ = train_one_run(model, train_loader, valid_loader, idx2tag, device, dual_encoder, epochs, verbose=verbose)
        test_metrics, _ = evaluate(model, test_loader, idx2tag, device, dual_encoder)
        test_metrics["seed"] = seed
        per_seed_results.append(test_metrics)
        del model
        torch.cuda.empty_cache()
    df = pd.DataFrame(per_seed_results)
    summary = {c: (df[c].mean(), df[c].std()) for c in df.columns if c != "seed"}
    return df, summary


def compare_configs(df_a, df_b, metric="span_f1"):
    """Iki config arasinda paired t-test (ayni seed sirasiyla eslenir)."""
    a, b = df_a[metric].values, df_b[metric].values
    n = min(len(a), len(b))
    t_stat, p_value = stats.ttest_rel(a[:n], b[:n])
    return {"metric": metric, "mean_a": a.mean(), "mean_b": b.mean(), "t_stat": t_stat, "p_value": p_value,
            "significant_at_0.05": p_value < 0.05}


## 8. Final Model — Proposed Architecture

Trains and evaluates the proposed configuration (BIOS scheme, duplicate subword propagation, convex-combination fusion) under the standard multi-seed protocol used throughout the paper.

In [ ]:
train_loader, valid_loader, test_loader, tag2idx, idx2tag = prepare_loaders_fusion("BIOS", "duplicate")

model_fn = lambda: FusionBiLSTMCRF(BERT_NAME, ELECTRA_NAME, len(tag2idx), fusion="convex")
df_final, summary_final = run_multi_seed(model_fn, train_loader, valid_loader, test_loader, idx2tag, device, dual_encoder=True)

df_final.to_csv(os.path.join(RESULTS_DIR, "final_model_convex_duplicate.csv"), index=False)
print("Per-seed sonuclar:")
print(df_final)
print()
print("Ozet (mean, std):")
for metric, (mean, std) in summary_final.items():
    print(f"  {metric}: {mean:.4f} +/- {std:.4f}")

## 9. Sonuçları Export Et

In [ ]:
final_summary = {
    "config": {"scheme": "BIOS", "propagation": "duplicate", "fusion": "convex"},
    "results": {m: list(v) for m, v in summary_final.items()},
}
with open(os.path.join(RESULTS_DIR, "final_model_summary.json"), "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2, ensure_ascii=False)
print("Kaydedildi:", os.path.join(RESULTS_DIR, "final_model_summary.json"))